In [ ]:
!pip install langchain-huggingface faiss-cpu langchain-groq -q
!pip install faiss-cpu langchain-community -q

In [ ]:
import os
import pandas as pd
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

os.environ['GROQ_API_KEY'] = 'your-key-here'


# ── Step 1: Load your real London property CSV ───────────────
df = pd.read_csv('/content/sample_data/kaggle_london_house_price_data.csv')

# Clean — keep rows with price and area
df = df[df['saleEstimate_currentPrice'].notna()]
df = df[df['saleEstimate_currentPrice'] > 0]
df = df.head(50)  # first 50 properties
print(f"✓ Loaded {len(df)} properties")

# ── Step 2: Convert each row to a LangChain Document ────────
documents = []
for _, row in df.iterrows():
    price    = row['saleEstimate_currentPrice']
    area     = row.get('outcode', 'Unknown')
    ptype    = row.get('propertyType', 'Unknown')
    sqm      = row.get('floorAreaSqM', 'Unknown')

    content = (
        f"Property in {area}. "
        f"Type: {ptype}. "
        f"Price: £{price:,.0f}. "
        f"Floor area: {sqm} sqm."
    )

    documents.append(Document(
        page_content=content,
        metadata={
            "area":  area,
            "price": price,
            "type":  str(ptype)
        }
    ))

print(f"✓ {len(documents)} documents created")

# ── Step 3: Build vector store ───────────────────────────────
embeddings  = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(documents, embeddings)
retriever   = vectorstore.as_retriever(search_kwargs={"k": 5})
print("✓ Vector store built")

# ── Step 4: RAG chain ────────────────────────────────────────
model  = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
parser = StrOutputParser()

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a London property investment advisor.
Answer using ONLY the property listings provided below.
Be specific — mention prices and locations from the data.
If the answer is not in the data, say 'I don't have that information.'
Never make up property details.

Listings:
{context}"""),
    ("human", "{question}")
])

def format_docs(docs):
    return "\n".join([
        f"- [{doc.metadata.get('area','Unknown')}]: {doc.page_content}"
        for doc in docs
    ])

rag_chain = (
    {"context":  retriever | format_docs,
     "question": RunnablePassthrough()}
    | rag_prompt | model | parser
)

print("✓ RAG chain ready")

# ── Step 5: Ask 5 investor questions ────────────────────────
questions = [
    "What types of properties are available?",
    "Which area has the most listings?",
    "What is the most expensive property and where is it?",
    "What is the cheapest property available?",
    "Which properties are best value based on price and size?"
]

print("\n" + "="*55)
print("London Property Q&A Assistant")
print("="*55)

for q in questions:
    print(f"\nQ: {q}")
    print(f"A: {rag_chain.invoke(q)}")

# ── Step 6: Stretch — save vector store to disk ──────────────
vectorstore.save_local("property_index")
print("\n✓ Vector store saved to property_index/")
print("  Load next time with:")
print("  vectorstore = FAISS.load_local('property_index', embeddings, allow_dangerous_deserialization=True)")